In [1]:
import numpy as np
import pandas as pd
import os, re, string, nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import gensim.downloader as api
from gensim.models import (
    Word2Vec,
    KeyedVectors
)
from gensim.utils import simple_preprocess

import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

import sklearn.metrics as metrics
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../../../data/news_data.csv")
df.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


In [3]:
subject_encoder = LabelEncoder()
df['subject'] = subject_encoder.fit_transform(df['subject'])
df['subject']

0        6
1        6
2        6
3        6
4        6
        ..
44893    1
44894    1
44895    1
44896    1
44897    1
Name: subject, Length: 44898, dtype: int64

In [4]:
df['date'] = pd.to_datetime(df['date'], errors='coerce', format='mixed')
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df.drop(columns=['date'], inplace=True)
df.head()

,title,text,subject,label,year,month,day
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,6,1,2017.0,12.0,31.0
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,6,1,2017.0,12.0,29.0
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,6,1,2017.0,12.0,31.0
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,6,1,2017.0,12.0,30.0
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,6,1,2017.0,12.0,29.0


In [19]:
x_train, x_test, y_train, y_test = train_test_split(
    df.drop(columns=['label']), df['label'], 
    train_size=0.5, 
    random_state=42
)
print("Data has been split.")

Data has been split.


In [18]:
# Our vocabulary
corpus = []
for text in df['text']:
    corpus.extend(text.split())

corpus = set(corpus)

print(
    "Size of Vocabulary:", 
    len(corpus)
)

Size of Vocabulary: 397481


In [20]:
stop_words = set(stopwords.words('english'))
stop_words.difference_update(['not', 'no', 'nor'])

lemmatizer = WordNetLemmatizer()

def text_preprocessor(text: str):
    text = re.sub(
        r'<.*?>', 
        '', 
        text
    ).lower()
    text = re.sub(
        r'http\S+|www\S+|https\S+', 
        '', 
        text, 
        flags=re.MULTILINE
    )
    text = re.sub(
        r" .*n't? ",
        " not ",
        text
    )
    
    return ' '.join([
        lemmatizer.lemmatize(token) 
        for token in word_tokenize(text) 
        if token not in stop_words and len(token) > 1
    ])

In [21]:
x_train['title'] = x_train['title'].apply(text_preprocessor)
x_train['text'] = x_train['text'].apply(text_preprocessor)
x_train.head()

,title,text,subject,year,month,day
7691,trump call '21st century glass-steagall bankin...,reuters u.s. republican presidential candidate...,6,2016.0,10.0,26.0
11073,white house back treasury view russia 's putin...,washington reuters u.s. treasury official asse...,6,2016.0,1.0,28.0
38307,obama radical agenda responsible state departm...,president barack obama called fight human traf...,0,2015.0,12.0,26.0
10935,clinton slam student loan servicer navient mis...,henniker n.h. reuters u.s. democratic presiden...,6,2016.0,2.0,6.0
32034,president trump first lady visit rep. steve sc...,class act president first lady visited rep. st...,5,2017.0,6.0,14.0


In [29]:
tokenizer = Tokenizer(num_words=50_000, oov_token='<oov>')
tokenizer.fit_on_texts(
    x_train['title']
    + x_train['text']
)

title_sequences = tokenizer.texts_to_sequences(x_train['title'])
x_train_title = np.concatenate([
    pad_sequences(
        title_sequences,
        maxlen=5,
        padding='pre',
        truncating='pre'
    ),
    pad_sequences(
        title_sequences,
        maxlen=5,
        padding='pre',
        truncating='pre'
    )
], axis=1)

text_sequences = tokenizer.texts_to_sequences(x_train['text'])
x_train_text = np.concatenate([
    pad_sequences(
        text_sequences,
        maxlen=128,
        padding='pre',
        truncating='pre'
    ),
    pad_sequences(
        text_sequences,
        maxlen=128,
        padding='pre',
        truncating='pre'
    )
], axis=1)

In [32]:
all_text = [
    str(t).split() 
    for t in x_train['title']
] + [
    str(t).split() 
    for t in x_train['text']
]

wv = Word2Vec(
    sentences=all_text,
    vector_size=128,
    max_final_vocab=50_000,
    epochs=25,
    workers=4
)
wv

In [38]:
vocab_size = 50_000
vector_size = 128

title_len = 10
text_len = 256

embedding_matrix = np.zeros((vocab_size, vector_size))

for word, i in tokenizer.word_index.items():
    if i < vocab_size and word in wv.wv:
        embedding_matrix[i] = wv.wv[word]

In [39]:
title_input = keras.Input(shape=(title_len,), name='title_input')
text_input = keras.Input(shape=(text_len,), name='text_input')

shared_embedding = keras.layers.Embedding(
    input_dim=vocab_size,
    output_dim=vector_size,
    weights=[embedding_matrix],
    trainable=True,
    name='word2vec_embedding'
)

title_embedded = shared_embedding(title_input)
text_embedded = shared_embedding(text_input)

title_lstm = keras.layers.LSTM(
    64, 
    return_sequences=False, 
    name='title_lstm'
)(title_embedded)
text_lstm = keras.layers.LSTM(
    128, 
    return_sequences=False, 
    name='text_lstm'
)(text_embedded)
merged = keras.layers.concatenate(
    [title_lstm, text_lstm], 
    name='feature_concatenation'
)

dense1 = keras.layers.Dense(
    64, 
    activation='relu', 
    name='hidden_dense'
)(merged)
dropout = keras.layers.Dropout(0.5, name='dropout')(dense1)
output = keras.layers.Dense(
    1, 
    activation='sigmoid', 
    name='output_layer'
)(dropout)

model = keras.Model(
    inputs=[title_input, text_input],
    outputs=output,
    name='text_title_lstm_model'
)

E0000 00:00:1793382072.336354    6831 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [40]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',  
    metrics=['accuracy']
)

model.summary()

Model: "text_title_lstm_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ title_input         │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_input          │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word2vec_embedding  │ (None, 256, 128)  │  6,400,000 │ title_input[0][0… │
│ (Embedding)         │                   │            │ text_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ title_lstm (LSTM)   │ (None, 64)        │     49,408 │ word2vec_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_lstm (LSTM)    │ (None, 128)       │    131,584 │ word2vec_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_concatenat… │ (None, 192)       │          0 │ title_lstm[0][0], │
│ (Concatenate)       │                   │            │ text_lstm[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hidden_dense        │ (None, 64)        │     12,352 │ feature_concaten… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ hidden_dense[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_layer        │ (None, 1)         │         65 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,593,409 (25.15 MB)

 Trainable params: 6,593,409 (25.15 MB)

 Non-trainable params: 0 (0.00 B)

In [41]:
model.fit(
    x=[x_train_title, x_train_text],  
    y=y_train,
    batch_size=32,
    epochs=5,
    validation_split=0.2
)

Epoch 1/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 614s 1s/step - accuracy: 0.9669 - loss: 0.0972 - val_accuracy: 0.9757 - val_loss: 0.0630
Epoch 2/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 489s 869ms/step - accuracy: 0.9904 - loss: 0.0291 - val_accuracy: 0.9860 - val_loss: 0.0438
Epoch 3/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 384s 682ms/step - accuracy: 0.9976 - loss: 0.0076 - val_accuracy: 0.9837 - val_loss: 0.0626
Epoch 4/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 383s 681ms/step - accuracy: 0.9994 - loss: 0.0023 - val_accuracy: 0.9862 - val_loss: 0.0636
Epoch 5/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 381s 677ms/step - accuracy: 0.9997 - loss: 0.0010 - val_accuracy: 0.9855 - val_loss: 0.0757


In [42]:
title_sequences = tokenizer.texts_to_sequences(x_test['title'])
x_test_title = np.concatenate([
    pad_sequences(
        title_sequences,
        maxlen=5,
        padding='pre',
        truncating='pre'
    ),
    pad_sequences(
        title_sequences,
        maxlen=5,
        padding='pre',
        truncating='pre'
    )
], axis=1)

text_sequences = tokenizer.texts_to_sequences(x_test['text'])
x_test_text = np.concatenate([
    pad_sequences(
        text_sequences,
        maxlen=128,
        padding='pre',
        truncating='pre'
    ),
    pad_sequences(
        text_sequences,
        maxlen=128,
        padding='pre',
        truncating='pre'
    )
], axis=1)

In [43]:
model.evaluate(
    [x_test_title, x_test_text],
    [y_test]
)

702/702 ━━━━━━━━━━━━━━━━━━━━ 87s 124ms/step - accuracy: 0.8972 - loss: 0.4735


[0.4735448658466339, 0.8972337245941162]